## Tools

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model

model_name = "claude-haiku-4-5-20251001"


model = init_chat_model("claude-haiku-4-5-20251001")
model

ChatAnthropic(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0', 'langchain-anthropic': '1.7.2'}}, profile={'name': 'Claude Haiku 4.5', 'release_date': '2025-10-15', 'last_updated': '2025-10-15', 'open_weights': False, 'max_input_tokens': 200000, 'max_output_tokens': 64000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': False, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_call_streaming': True}, model='claude-haiku-4-5-20251001', max_tokens=64000, anthropic_api_url='https://api.anthropic.com', anthropic_api_key=SecretStr('**********'), model_kwargs={})

In [5]:
from langchain.tools import tool
from langchain.agents import create_agent



@tool
def get_weather(location:str) -> str:
    """Get the current weather for a given location."""
    # Here you would implement the logic to get the weather for the location
    return f"The current weather in {location} is sunny with a temperature of 25°C."


# this tools need to be bind with llm
#1 approach way 
model_with_tools = model.bind_tools([get_weather])

# 2. apporach 
def get_weather(city : str)->str:
    "Get weather information for a given city."
    return f"the weather in {city} is sunny and 25°C."

agent = create_agent(
    model = model_name,
    tools = [get_weather],
    system_prompt = "You are a helpful assistant.",
    # verbose = True
)


In [10]:
response = model_with_tools.invoke("What is the weather in New York City?")
print(response.content)

for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"Input used: {tool_call['args']}")

[{'id': 'toolu_014RK5q6pmkjuDzjszGS297h', 'caller': {'type': 'direct'}, 'input': {'location': 'New York City'}, 'name': 'get_weather', 'type': 'tool_use', 'toolset_name': None}]
Tool called: get_weather
Input used: {'location': 'New York City'}


In [13]:
## Tool execution steps

# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

AttributeError: 'function' object has no attribute 'invoke'